<a href="https://colab.research.google.com/github/AltaCedeno/ALTA/blob/main/mapaAreaRevisores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Carga del archivo CSV

Primero, cargaremos tu archivo CSV en Google Colab. Se te pedirá que selecciones el archivo desde tu sistema local.

In [4]:
import pandas as pd
from google.colab import files
import io

print("Por favor, sube tu archivo CSV:")
uploaded = files.upload()

for fn in uploaded.keys():
  print('Archivo "{name}" de {length} bytes subido con éxito'.format(
      name=fn, length=len(uploaded[fn])))
  csv_filename = fn

# Lee el archivo CSV en un DataFrame de pandas
df = pd.read_csv(io.StringIO(uploaded[csv_filename].decode('utf-8')))

print(f"Se ha cargado el archivo: {csv_filename}")
display(df.head())

Por favor, sube tu archivo CSV:


Saving TablaProyectoFinal.csv to TablaProyectoFinal (1).csv
Archivo "TablaProyectoFinal (1).csv" de 11023 bytes subido con éxito
Se ha cargado el archivo: TablaProyectoFinal (1).csv


,id_trampa,code_revisor,latitud,longitud
0,LA9800,1,18.684451,-68.632660
1,LA9896,1,18.739182,-68.549148
2,LA1019,1,18.743185,-68.548795
3,LA969,1,18.689490,-68.563353
4,LA630,1,18.766365,-68.548127


### 2. Identificación de Columnas

Ahora, identificaremos las columnas del DataFrame, prestando especial atención a `id_trampa`, `longitud`, `latitud` y `code_revisor`.

In [8]:
print("Información general del DataFrame:")
df.info()

print("\nColumnas del DataFrame:")
print(df.columns)

# Verificar la presencia de las columnas requeridas con los nombres reales del CSV
required_columns = ['id_trampa', 'longitud', 'latitud', 'code_revisor']
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print(f"\n¡Advertencia! Faltan las siguientes columnas requeridas: {', '.join(missing_columns)}")
else:
    print("\nTodas las columnas requeridas (id_trampa, longitud, latitud, code_revisor) están presentes.")

# Muestra algunas filas para las columnas clave con los nombres correctos
display(df[['id_trampa', 'longitud', 'latitud', 'code_revisor']].head())

Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 355 entries, 0 to 354
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_trampa     355 non-null    object 
 1   code_revisor  355 non-null    int64  
 2   latitud       355 non-null    float64
 3   longitud      355 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 11.2+ KB

Columnas del DataFrame:
Index(['id_trampa', 'code_revisor', 'latitud', 'longitud'], dtype='object')

Todas las columnas requeridas (id_trampa, longitud, latitud, code_revisor) están presentes.


,id_trampa,longitud,latitud,code_revisor
0,LA9800,-68.632660,18.684451,1
1,LA9896,-68.549148,18.739182,1
2,LA1019,-68.548795,18.743185,1
3,LA969,-68.563353,18.689490,1
4,LA630,-68.548127,18.766365,1


### 3. Generación del archivo KML con Polígonos Coloreados

Crearemos un archivo KML donde cada valor único en la columna `code_revisor` representará un polígono distinto. Para generar los polígonos, se calculará la envoltura convexa (convex hull) de los puntos (`latitud`, `longitud`) asociados a cada `code_revisor`. Los polígonos serán coloreados para diferenciarlos.

In [11]:
# Instalar librerías necesarias
!pip install simplekml scipy

import simplekml
from scipy.spatial import ConvexHull
import numpy as np

kml = simplekml.Kml()

# Asegurarse de que las columnas de latitud y longitud sean numéricas
df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')

# Eliminar filas con valores NaN en latitud o longitud que podrían resultar de la conversión
df_cleaned = df.dropna(subset=['latitud', 'longitud']).copy()

# Definir una lista de colores para los polígonos (puedes añadir más si tienes más de 3 code_revisor)
colors = [
    simplekml.Color.red,
    simplekml.Color.blue,
    simplekml.Color.green,
    simplekml.Color.yellow,
    simplekml.Color.orange,
    simplekml.Color.purple
]
color_index = 0

# Agrupar por la columna 'code_revisor'
grouped_by_revisor = df_cleaned.groupby('code_revisor')

# Corregido: Usar 'grouped_by_revisor' en lugar de 'group_by_revisor'
print(f"Generando KML para {len(grouped_by_revisor.groups)} grupos de 'code_revisor'...")

for revisor_code, group in grouped_by_revisor:
    if len(group) < 3:
        print(f"El grupo '{revisor_code}' tiene menos de 3 puntos ({len(group)}). No se puede formar un polígono. Se añadirán solo puntos.")
        folder = kml.newfolder(name=f"Grupo: {revisor_code} (Puntos)")
        for idx, row in group.iterrows():
            pnt = folder.newpoint(name=str(row['id_trampa']))
            pnt.coords = [(row['longitud'], row['latitud'])]
            pnt.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/pal2/icon18.png'
            pnt.style.iconstyle.color = colors[color_index % len(colors)]
            pnt.style.labelstyle.color = colors[color_index % len(colors)]
        color_index += 1
        continue

    coords = group[['longitud', 'latitud']].values

    # Calcular la envoltura convexa para formar el polígono
    hull = ConvexHull(coords)

    # Obtener los vértices de la envoltura convexa en orden
    polygon_coords = [tuple(coords[vertex]) for vertex in hull.vertices]

    # Cerrar el polígono repitiendo el primer punto
    polygon_coords.append(polygon_coords[0])

    # Crear una carpeta para cada grupo de code_revisor
    folder = kml.newfolder(name=f"Grupo: {revisor_code}")

    # Crear el polígono
    pol = folder.newpolygon(name=f"Polígono {revisor_code}")
    pol.outerboundaryis = polygon_coords

    # Asignar color al polígono
    pol.style.polystyle.color = colors[color_index % len(colors)] # Usa el siguiente color de la lista
    pol.style.polystyle.fill = 1 # Rellena el polígono
    pol.style.polystyle.outline = 1 # Dibuja el contorno
    pol.style.linestyle.width = 3 # Ancho del contorno

    # Opcional: añadir los puntos individuales también, para visualización
    for idx, row in group.iterrows():
        pnt = folder.newpoint(name=str(row['id_trampa']))
        pnt.coords = [(row['longitud'], row['latitud'])]
        pnt.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/pal2/icon18.png'
        pnt.style.iconstyle.color = colors[color_index % len(colors)]
        pnt.style.labelstyle.color = colors[color_index % len(colors)]

    color_index += 1 # Avanza al siguiente color

# Guardar el archivo KML
kml_filename = 'poligonos_coordenadas.kml'
kml.save(kml_filename)
print(f"Archivo KML guardado como '{kml_filename}'")

Generando KML para 3 grupos de 'code_revisor'...
Archivo KML guardado como 'poligonos_coordenadas.kml'


### 4. Descargar el archivo KML

Usa el siguiente código para descargar el archivo KML generado a tu ordenador.

In [1]:
!pip install simplekml scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for simplekml: filename=simplekml-1.3.6-py3-none-any.whl size=65860 sha256=cf8dfabf2423bb581db3ed006cf05885c0a087e10b78561eaeabeb57bc743bb7
  Stored in directory: /root/.cache/pip/wheels/83/ee/f2/65cecfd948f1429ead035fd6d56bc6bd6574a636ddc4d65cbd
Successfully built simplekml


In [7]:
from google.colab import files

files.download(poligonos_coordenadas.kml)

NameError: name 'kml_filename' is not defined

In [13]:
from google.colab import files

# Usa la variable kml_filename que contiene el nombre del archivo KML
files.download(kml_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 5. Visualización de datos en un mapa interactivo

Usaremos `folium` para crear un mapa interactivo con los puntos de latitud y longitud.

In [14]:
# Instalar folium si aún no está instalado
!pip install folium

import folium

# Calcular el centro del mapa
center_lat = df['latitud'].mean()
center_lon = df['longitud'].mean()

# Crear un mapa interactivo con folium
m = folium.Map(location=[center_lat, center_lon], zoom_start=9)

# Añadir un marcador para cada punto en el DataFrame
for index, row in df.iterrows():
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=f"ID Trampa: {row['id_trampa']}, Revisor: {row['code_revisor']}",
        tooltip=row['id_trampa']
    ).add_to(m)

# Mostrar el mapa
m